# Lesson 1: Introduction to Transformers

The [Transformers](https://github.com/huggingface/transformers) library by Hugging Face is the de facto standard for working with open-source language models. It provides:

- **[Model Hub](https://huggingface.co/models)**: Access to thousands of pre-trained models
- **[Auto classes](https://huggingface.co/docs/transformers/en/model_doc/auto)**: Automatic model class detection based on architecture
- **Tokenizers**: Fast tokenization with [chat template](https://huggingface.co/docs/transformers/en/chat_templating) support
- **[Generation](https://huggingface.co/docs/transformers/en/generation_strategies)**: Flexible text generation with multiple decoding strategies
- **[Serving](https://huggingface.co/docs/transformers/en/serve-cli/serving)**: An OpenAI-compatible API server via the CLI

In this lesson, we'll load and run **Qwen3-0.6B**. We'll cover:

1. Loading a model with `from_pretrained`
2. Generating text with `generate`
3. Chat-style generation with chat templates
4. Continuous batching with `generate_batch`
5. Serving with `transformers serve`

## Loading a Model

The key classes for working with LLMs in Transformers are:

- `AutoModelForCausalLM` — automatically detects and loads the right model class for causal (left-to-right) language models
- `AutoTokenizer` — loads the matching tokenizer

The `from_pretrained` method downloads the model weights from the Hugging Face Hub (or loads from cache) and instantiates the model.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print(f"Model class: {type(model).__name__}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")

## Generating Text

Text generation in Transformers follows a **tokenize → generate → decode** pipeline:

1. **Tokenize** the input text into token IDs
2. **Generate** new token IDs autoregressively
3. **Decode** the token IDs back into text

The [`generate`](https://huggingface.co/docs/transformers/en/generation_strategies) method supports many parameters via `GenerationConfig`, including:
- `max_new_tokens` — maximum number of tokens to generate
- `temperature` — controls randomness (0 = greedy, higher = more random)
- `top_p` — nucleus sampling threshold
- `do_sample` — whether to sample (vs greedy decoding)

In [ ]:
prompt = "The key ingredients for a great pizza are"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
print(f"Input token IDs: {inputs.input_ids[0].tolist()[:10]}... ({inputs.input_ids.shape[1]} tokens)")

output_ids = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
)

# Decode only the newly generated tokens
generated_text = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"\nPrompt: {prompt}")
print(f"Generated: {generated_text}")

## Chat-style Generation

Instruction-tuned models (like Qwen3) expect structured input in a specific chat format. Rather than crafting this format manually, we can use [`apply_chat_template`](https://huggingface.co/docs/transformers/en/chat_templating) to convert a list of messages into the model's expected format.

Messages follow the OpenAI-style format:
```python
[{"role": "user", "content": "Hello!"}, {"role": "assistant", "content": "Hi!"}]
```

In [ ]:
messages = [
    {"role": "user", "content": "What are the key ingredients of great pizza?"}
]

# First, let's see what the chat template actually produces
templated_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
print("Templated text:")
print(templated_text)
print()

# Now tokenize and generate
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

output_ids = model.generate(**inputs, max_new_tokens=1000)

# Decode only the assistant's response
response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

## Continuous Batching with `generate_batch`

When processing multiple prompts, naive batching pads all sequences to the same length, wasting compute on padding tokens. **[Continuous batching](https://huggingface.co/blog/continuous_batching)** solves this by:

- Processing sequences independently (no padding needed)
- As shorter sequences finish, new ones can join the batch immediately
- Using paged attention for efficient KV cache management

The [`generate_batch`](https://huggingface.co/docs/transformers/en/continuous_batching) method takes a list of token ID lists (not padded tensors) and handles scheduling internally. This is the same core idea that production serving engines like vLLM use.

In [ ]:
from transformers import GenerationConfig
import torch

# --- Apple Silicon / MPS workarounds for continuous batching on CPU ---
# Both issues only reproduce when PyTorch sees MPS as available; on Linux/CUDA
# or CPU-only builds these patches are unnecessary and we skip them.
if torch.backends.mps.is_available():
    # 1) pin_memory path: transformers forces pin_memory=True whenever any
    #    accelerator is visible. On Mac that pins CPU tensors via MPS, and
    #    `zero_()` then dispatches to MPS (which has no kernel for it).
    #    Hiding MPS from the heuristic forces pin_memory=False.
    import transformers.generation.continuous_batching.input_outputs as _cb_io
    _cb_io.get_available_devices = lambda: frozenset({"cpu"})

    # 2) available-memory path: on CPU, cache auto-sizing computes
    #    `available = total - Process.rss`. On macOS, RSS counts mmap'd
    #    safetensors pages and reclaimable file cache, so RSS ~= total RAM and
    #    `available` collapses to a few hundred KB — which trips the
    #    MemoryError guard. The right metric is `psutil.virtual_memory().available`,
    #    which excludes reclaimable pages. `cache.py` imports the breakdown
    #    helper by name, so we patch the alias on the `cache` module (patching
    #    `requests` has no effect).
    import psutil
    import transformers.generation.continuous_batching.cache as _cb_cache

    def _cpu_memory_breakdown():
        vmem = psutil.virtual_memory()
        used = vmem.total - vmem.available
        return torch.device("cpu"), vmem.total, used, used

    _cb_cache.get_device_and_memory_breakdown = _cpu_memory_breakdown

conversations = [
    [{"role": "user", "content": "What is the capital of France?"}],
    [{"role": "user", "content": "Write a haiku about programming."}],
    [{"role": "user", "content": "What are the three laws of thermodynamics?"}],
    [{"role": "user", "content": "Describe a galaxy far, far away."}],
]

# Apply the chat template per conversation — no padding, just a list of token IDs
# per request, which is what generate_batch expects.
inputs = [
    tokenizer.apply_chat_template(conv, add_generation_prompt=True, tokenize=True)["input_ids"]
    for conv in conversations
]

generation_config = GenerationConfig(
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
)

outputs = model.generate_batch(
    inputs=inputs,
    generation_config=generation_config,
)

for request_id, output in outputs.items():
    response = tokenizer.decode(output.generated_tokens, skip_special_tokens=True)
    print(f"[{request_id}] {response}\n")

## Cleaning Up

The in-memory model holds the weights and KV cache. When you're done with offline inference, delete it to free that memory — especially important before starting the server, which will load its own copy.

In [ ]:
import gc

del model, tokenizer
gc.collect()

## The `transformers serve` CLI

Transformers includes a built-in server that exposes an **OpenAI-compatible API**. This means you can use the same client code (e.g., the `openai` Python package) to talk to your local model.

Key flags:

- `--device cpu` — run on CPU (use `cuda` for GPU)
- `--dtype auto` — use the model's native precision
- `--port 8000` — server port (default)
- `--continuous_batching` — enable continuous batching with paged attention*

_*We are not using this in this lesson because of the MPS workarounds currently required._

The server exposes these endpoints:

- `POST /v1/chat/completions` — chat completions
- `GET /v1/models` — list loaded models
- `GET /health` — health check

See the [full `transformers serve` documentation](https://huggingface.co/docs/transformers/en/serve-cli/serving) for all available options.

Let's start it as a background process so we can query it from this notebook.

In [ ]:
import os
import subprocess
import time
import urllib.request

# On Apple Silicon, `transformers serve` can still route some ops (e.g. torch.histc
# in the MoE grouped-MM path) through MPS, where integer kernels aren't implemented.
# PYTORCH_ENABLE_MPS_FALLBACK=1 transparently falls those ops back to CPU.
server_env = {**os.environ, "PYTORCH_ENABLE_MPS_FALLBACK": "1"}

server = subprocess.Popen(
    ["transformers", "serve", MODEL_NAME, "--device", "cpu"],
    env=server_env,
)
print(f"Started transformers serve with PID: {server.pid}")

# Wait for the server to be ready
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health")
        print("Server is ready!")
        break
    except Exception:
        time.sleep(1)
else:
    print("Server did not start in time")

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "What is a mixture-of-experts model?"}],
    max_tokens=100,
)

print(response.choices[0].message.content)

## Stopping the Server

When you're done, terminate the server process using the PID we captured earlier.

In [ ]:
server.terminate()
server.wait()
print(f"Server (PID {server.pid}) stopped.")

## Summary

In this lesson we covered:

- **`from_pretrained`** — loading models and tokenizers from the Hugging Face Hub
- **`generate`** — the tokenize → generate → decode pipeline for text generation
- **Chat templates** — structured input for instruction-tuned models
- **`generate_batch`** — continuous batching for efficient multi-prompt inference
- **`transformers serve`** — an OpenAI-compatible API server

In the next lesson, we'll explore **vLLM** — a high-throughput inference engine designed specifically for serving LLMs at scale.